# Dynamic Topic Modeling: Health Misinformation Over Time

**Goal**: Track how health-related misinformation topics evolve across time using LDA and BERTopic, comparing topic evolution across 2-year windows.

**Temporal data**: Uses real tweet timestamps from `Truth_Seeker_Model_Dataset_With_TimeStamps.xlsx` ( `created_at`： 2008–2022). 

---
## 0. Setup & Imports

In [ ]:
import re
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from collections import defaultdict, Counter

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Scikit-learn
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from scipy.spatial.distance import jensenshannon

# Visualization
from wordcloud import WordCloud

warnings.filterwarnings('ignore')

# Download NLTK data
for resource in ['stopwords', 'wordnet', 'punkt', 'punkt_tab', 'averaged_perceptron_tagger']:
    nltk.download(resource, quiet=True)

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
RANDOM_STATE = 42

# Reproducibility controls for stochastic topic models.
import random
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## 1. Data Loading & Merging

In [ ]:
import os
DATA_DIR = "data/"
PLOT_DIR = "output/"

# Load feature dataset (engineered NLP features + text)
features_df = pd.read_csv(DATA_DIR + "Features_For_Traditional_ML_Techniques.csv",
                          index_col=0)

# Load base metadata dataset
meta_df = pd.read_csv(DATA_DIR + "Truth_Seeker_Model_Dataset.csv",
                      index_col=0)

# ── Load timestamped dataset ─────────────────────────────────────────────────
# File is an Excel (.xlsx); place it in data/ or update the path below.
TIMESTAMPS_PATH = os.path.expanduser(
    "~/Downloads/TruthSeeker2023/Truth_Seeker_Model_Dataset_With_TimeStamps 1.xlsx"
)
ALT_TIMESTAMPS_PATH = DATA_DIR + "Truth_Seeker_Model_Dataset_With_TimeStamps.xlsx"

if os.path.exists(ALT_TIMESTAMPS_PATH):
    ts_df = pd.read_excel(ALT_TIMESTAMPS_PATH, index_col=0)
    print(f"✓ Loaded timestamps from {ALT_TIMESTAMPS_PATH}")
elif os.path.exists(TIMESTAMPS_PATH):
    ts_df = pd.read_excel(TIMESTAMPS_PATH, index_col=0)
    print(f"✓ Loaded timestamps from {TIMESTAMPS_PATH}")
    print("  Tip: copy this file to data/ for cleaner project layout.")
else:
    ts_df = None
    print("⚠ Timestamped file not found — Section 3 will use year-proxy fallback.")

# Parse the timestamp column
if ts_df is not None:
    ts_df['timestamp'] = pd.to_datetime(
        ts_df['timestamp'],
        format='%a %b %d %H:%M:%S +0000 %Y',
        utc=True,
        errors='coerce'
    )
    print(f"  Date range: {ts_df['timestamp'].min().date()} → {ts_df['timestamp'].max().date()}")
    print(f"  Year distribution:\n{ts_df['timestamp'].dt.year.value_counts().sort_index().to_string()}")

#  Merge all sources 
df = features_df.copy()
df['manual_keywords'] = meta_df['manual_keywords'].values
df['author']          = meta_df['author'].values
df['label_5']         = meta_df['5_label_majority_answer'].values

if ts_df is not None:
    # Align timestamps by the original row index, not by position. The timestamped
    # file has a few extra rows, so positional assignment can attach COVID-era
    # tweets to pre-2020 timestamps.
    ts_aligned = ts_df.reindex(df.index)
    statement_match = (df['statement'].astype(str) == ts_aligned['statement'].astype(str)).mean()
    print(f"  Timestamp alignment statement match: {statement_match:.2%}")
    df['timestamp'] = ts_aligned['timestamp']

print(f"\nMerged shape: {df.shape}")
print(f"Unique statements: {df['statement'].nunique()}")
print(f"Label distribution:\n{df['majority_target'].value_counts()}")
df.head(5)

## 2. Data Cleaning & Health Filtering

In [ ]:
# Drop nulls in core columns
df = df.dropna(subset=['statement', 'tweet'])

# Ensure binary label is clean
df['label'] = df['BinaryNumTarget'].fillna(0).astype(int)  # 1=True/Credible, 0=False/Misinformation

print(f"After null drop: {df.shape}")
print(f"Label counts: {df['label'].value_counts().to_dict()}")

### Health-topic filtering

Applied to `statement` (the claim itself) to avoid false positives from tweet context.

In [ ]:
HEALTH_TERMS = [
    # Vaccines & immunization
    r'vaccin', r'immuniz', r'ivermectin', r'hydroxychloroquine', r'booster',
    r'pfizer', r'moderna', r'astrazeneca', r'johnson',
    # COVID / pandemic
    r'covid', r'coronavirus', r'pandemic', r'quarantine', r'lockdown',
    r'mask', r'ppe', r'social.distanc',
    # General health
    r'\bhealth\b', r'\bdisease\b', r'\bvirus\b', r'\bflu\b',
    r'\bcancer\b', r'\bdrug\b', r'hospital', r'\bCDC\b', r'\bFDA\b',
    r'\bWHO\b', r'\bFauci\b', r'\bpublic health\b', r'convalescent',
    r'\bplasma\b', r'asymptomatic', r'herd immunity', r'natural immunity',
]

pattern = '|'.join(HEALTH_TERMS)
health_mask = df['statement'].str.contains(pattern, case=False, na=False, regex=True)
health_df = df[health_mask].copy()

print(f"Health-related rows: {len(health_df):,} / {len(df):,} ({100*len(health_df)/len(df):.1f}%)")
print(f"Unique health statements: {health_df['statement'].nunique()}")
print(f"\nTop health topics (manual_keywords):")
health_df['manual_keywords'].value_counts().head(20)

In [ ]:
print("Health subset — label distribution:")
print(health_df['label'].value_counts())
print(f"\nMisinformation rate: {(health_df['label']==0).mean()*100:.1f}%")

## 3. Temporal Chunking

In [ ]:
def extract_year_proxy(text, min_year=2015, max_year=2023):
    """Fallback: most recent plausible year found in tweet text."""
    years = re.findall(r'\b(20[1-2]\d)\b', str(text))
    years = [int(y) for y in years if min_year <= int(y) <= max_year]
    return max(years) if years else np.nan


if 'timestamp' in health_df.columns and health_df['timestamp'].notna().sum() > 0:
    health_df['tweet_year'] = pd.to_datetime(health_df['timestamp'], utc=True, errors='coerce').dt.year
    print(f"Using real timestamps — {health_df['tweet_year'].notna().sum():,} valid")
else:
    health_df['tweet_year'] = health_df['tweet'].apply(extract_year_proxy)
    print(f"Using year-proxy fallback — {health_df['tweet_year'].notna().sum():,} valid")

print(health_df['tweet_year'].value_counts().sort_index())

### 3b. Assign 2-year temporal windows and filter for fake news only

In [ ]:
WINDOW_BINS   = [2014, 2016, 2018, 2020, 2022]
WINDOW_LABELS = ['2015-2016', '2017-2018', '2019-2020', '2021-2022']

health_df['time_window'] = pd.cut(
    health_df['tweet_year'],
    bins=WINDOW_BINS,
    labels=WINDOW_LABELS,
    right=True
)

temporal_df = health_df.dropna(subset=['time_window']).copy()

temporal_df = temporal_df[temporal_df['majority_target'].astype(str).str.strip() == 'False'].copy()

temporal_df['time_window'] = temporal_df['time_window'].astype(str)

print(f"majority_target dtype: {health_df['majority_target'].dtype}")
print(f"Sample values: {health_df['majority_target'].unique()[:4]}")
print(f"\nFake news documents per 2-year window:")
print(temporal_df['time_window'].value_counts().sort_index())
print(f"\nTotal fake news docs: {len(temporal_df):,}")

fig, ax = plt.subplots(figsize=(6, 4))
window_counts = temporal_df['time_window'].value_counts().sort_index()
ax.bar(window_counts.index, window_counts.values, color='steelblue', edgecolor='white')
ax.set_title('Fake News Tweet Count per Window (majority_target=False)')
ax.set_xlabel('Time Window')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=20)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/01_fake_news_counts.png", dpi=150, bbox_inches='tight')
plt.show()

## 4. Text Preprocessing

 lowercase → strip URLs/@mentions → remove punctuation → tokenize → stopwords → lemmatize → keep alpha tokens ≥ 3 chars.

In [ ]:
lemmatizer = WordNetLemmatizer()
base_stops = set(stopwords.words('english'))

DOMAIN_STOPS = {
    'said', 'say', 'says', 'people', 'would', 'could', 'also', 'one', 'two',
    'get', 'got', 'go', 'going', 'like', 'make', 'know', 'think', 'time',
    'year', 'new', 'many', 'even', 'way', 'need', 'take', 'come', 'see',
    'well', 'good', 'right', 'day', 'fact', 'claim', 'true', 'false',
    'report', 'show', 'find', 'use', 'amp',
    'https', 'http', 'twitter', 'tweet', 'rt', 'via', 'video', 'watch',
    'read', 'article', 'link', 'share', 'post', 'news'
}

ALL_STOPS = base_stops | DOMAIN_STOPS

def clean_text(text):
    """Full preprocessing pipeline: returns a list of clean tokens."""
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()

    tokens = word_tokenize(text)
    tokens = [
        lemmatizer.lemmatize(tok)
        for tok in tokens
        if tok.isalpha()
        and len(tok) >= 3
        and tok not in ALL_STOPS
    ]
    return tokens


print("Preprocessing health documents...")
temporal_df['tokens'] = temporal_df['tweet'].apply(clean_text)
temporal_df['clean_text'] = temporal_df['tokens'].apply(lambda toks: ' '.join(toks))

sample_row = temporal_df.iloc[0]
print(f"\nOriginal tweet: {sample_row['tweet'][:80]}...")
print(f"Clean tokens (first 15): {sample_row['tokens'][:15]}")
print(f"\nTotal documents in temporal corpus: {len(temporal_df):,}")

In [ ]:
## Top terms per window
print("Top 20 terms per time window (raw frequency):")
print("=" * 60)
for win in WINDOW_LABELS:
    win_tokens = temporal_df[temporal_df['time_window'] == win]['tokens']
    flat_tokens = [tok for toks in win_tokens for tok in toks]
    top = Counter(flat_tokens).most_common(20)
    terms = [f"{w}({n})" for w, n in top]
    print(f"\n[{win}] — {len(win_tokens):,} docs")
    print("  " + ', '.join(terms))

## 5. LDA Topic Modeling

Single LDA (K=6 topics) trained on **all** fake-news documents combined. Per-window topic prevalence is then computed by applying the global model to each window's documents, so topic identities are consistent across windows.

In [ ]:
N_TOPICS = 6
N_TOP_WORDS = 10
MAX_FEATURES = 3000  # vocabulary cap

# ── Train ONE global LDA on all fake-news documents ──────────────────────────
all_fake_docs = temporal_df['clean_text'].tolist()

cv_global = CountVectorizer(
    max_features=MAX_FEATURES,
    min_df=5,
    max_df=0.85,
    ngram_range=(1, 2)
)
dtm_global = cv_global.fit_transform(all_fake_docs)

lda_global = LatentDirichletAllocation(
    n_components=N_TOPICS,
    max_iter=20,
    learning_method='batch',  # deterministic full-batch updates for reproducibility
    random_state=RANDOM_STATE,
    n_jobs=1                  # avoid non-deterministic parallel update ordering
)
doc_topic_global = lda_global.fit_transform(dtm_global)

vocab = cv_global.get_feature_names_out()
top_words_global = []
for topic_vec in lda_global.components_:
    top_idx = topic_vec.argsort()[:-N_TOP_WORDS - 1:-1]
    top_words_global.append([vocab[i] for i in top_idx])

perp = lda_global.perplexity(dtm_global)
print(f"Global LDA | All fake-news docs: {len(all_fake_docs):,} | Perplexity: {perp:.1f}")

# ── Slice per-window doc-topic matrices (model/vocabulary are shared) ─────────
lda_results = {}
temporal_reset = temporal_df.reset_index(drop=True)

for win in WINDOW_LABELS:
    win_idx = temporal_reset.index[temporal_reset['time_window'] == win].tolist()
    if len(win_idx) < 50:
        print(f"[{win}] Too few docs ({len(win_idx)}), skipping.")
        continue
    lda_results[win] = {
        'model':            lda_global,
        'vectorizer':       cv_global,
        'top_words':        top_words_global,
        'doc_topic_matrix': doc_topic_global[win_idx],
        'n_docs':           len(win_idx),
    }
    print(f"[{win}] Docs: {len(win_idx):,}")

print("\nLDA fitting complete.")

### 5a. Top words per topic per window

In [ ]:
print("Global LDA — Top 10 terms per topic (trained on all fake-news documents):\n")
print("=" * 60)
for t_idx, words in enumerate(top_words_global):
    print(f"  Topic {t_idx+1}: {', '.join(words)}")

print("\n\nTopic prevalence per time window")
print("(% of docs whose dominant topic is each topic)\n")

header = f"{'Topic':<10}" + "".join(f"{w:>14}" for w in WINDOW_LABELS if w in lda_results)
print(header)
print("-" * len(header))

windows_with_data = [w for w in WINDOW_LABELS if w in lda_results]
dominant_per_window = {}
for win in windows_with_data:
    dtm = lda_results[win]['doc_topic_matrix']
    dominant_per_window[win] = np.argmax(dtm, axis=1)  # 0-based topic index

for t_idx in range(N_TOPICS):
    row = f"Topic {t_idx+1:<4}"
    for win in windows_with_data:
        dom = dominant_per_window[win]
        pct = (dom == t_idx).sum() / len(dom) * 100
        row += f"{pct:>13.1f}%"
    print(row)

print("-" * len(header))
totals = f"{'N docs':<10}"
for win in windows_with_data:
    totals += f"{lda_results[win]['n_docs']:>14,}"
print(totals)

### 5b. Mean topic weight distribution per window

In [ ]:
windows_available = [w for w in WINDOW_LABELS if w in lda_results]
n_win = len(windows_available)

fig, axes = plt.subplots(1, n_win, figsize=(4 * n_win, 4), sharey=True)
if n_win == 1:
    axes = [axes]

colors = plt.cm.tab10(np.linspace(0, 1, N_TOPICS))

for ax, win in zip(axes, windows_available):
    mean_weights = lda_results[win]['doc_topic_matrix'].mean(axis=0)
    topic_labels = [f"T{i+1}" for i in range(N_TOPICS)]
    bars = ax.bar(topic_labels, mean_weights, color=colors, edgecolor='white')
    ax.set_title(win, fontsize=10, fontweight='bold')
    ax.set_xlabel('Topic')
    ax.set_ylim(0, 0.35)

axes[0].set_ylabel('Mean Topic Weight')
plt.suptitle('LDA: Mean Topic Weight Distribution per 2-Year Window', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/02_lda_topic_weights.png", dpi=150, bbox_inches='tight')
plt.show()

## 6. BERTopic Dynamic Topic Modeling (DTM)

BERTopic uses transformer embeddings + UMAP + HDBSCAN to discover topics, then `topics_over_time` tracks how each topic's frequency evolves across the 4 time windows — without fitting a separate model per period, so topic identities are consistent across windows.

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

docs_bt       = temporal_df['clean_text'].tolist()
timestamps_bt = temporal_df['time_window'].tolist()   # 4 pre-defined windows as timestamps

umap_model       = UMAP(n_neighbors=15, n_components=5, min_dist=0.0,
                        metric='cosine', random_state=RANDOM_STATE)
hdbscan_model    = HDBSCAN(min_cluster_size=50, metric='euclidean',
                            cluster_selection_method='eom', prediction_data=True)
vectorizer_model = CountVectorizer(stop_words='english', min_df=5, ngram_range=(1, 2))

topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    nr_topics=N_TOPICS,
    verbose=True
)

bt_topics, _ = topic_model.fit_transform(docs_bt)

n_real_topics = len([t for t in set(bt_topics) if t != -1])
print(f"Topics found (excl. outliers): {n_real_topics}")
print(f"Outlier docs (-1):             {sum(t == -1 for t in bt_topics):,}")
topic_model.get_topic_info()

### 6a. Top terms per BERTopic topic

In [ ]:
print("BERTopic — Top 10 terms per topic:")
print("=" * 60)
for topic_id, words in sorted(topic_model.get_topics().items()):
    if topic_id == -1:
        continue
    terms = ', '.join([w for w, _ in words[:10]])
    count = sum(t == topic_id for t in bt_topics)
    print(f"  Topic {topic_id + 1:2d} ({count:,} docs): {terms}")

### 6b. Dynamic topic evolution over time

`topics_over_time` computes, for each topic, how frequently it appears in each time window using the same model fitted on all data — topic identities are consistent across windows.

In [ ]:

# BERTopic requires numeric or parseable datetime timestamps.
# Map window labels to their start year (int) so pd.to_datetime is not invoked,
# then remap the result back to the original label strings for all downstream cells.
WINDOW_TO_YEAR = {
    '2015-2016': 2015,
    '2017-2018': 2017,
    '2019-2020': 2019,
    '2021-2022': 2021,
}
YEAR_TO_WINDOW = {v: k for k, v in WINDOW_TO_YEAR.items()}

timestamps_int = temporal_df['time_window'].map(WINDOW_TO_YEAR).tolist()

# Verify alignment: every doc must have a mapped timestamp (no NaN)
n_unmapped = sum(pd.isna(t) for t in timestamps_int)
print(f"Docs with unmapped timestamp: {n_unmapped} (should be 0)")
print(f"Timestamp value counts: {pd.Series(timestamps_int).value_counts().sort_index().to_dict()}")

# global_tuning=True would average each window's c-TF-IDF with the GLOBAL model,
# which is overwhelmingly vaccine/COVID-era (2021-22) and bleeds those terms into
# the sparse 2015-2016 window. Disable both tuning flags for window-faithful words.
topics_over_time = topic_model.topics_over_time(
    docs_bt,
    timestamps_int,
    global_tuning=False,
    evolution_tuning=False,
)

bt_valid = topics_over_time[topics_over_time['Topic'] != -1].copy()
# Shift topic IDs to 1-based for display; model internals remain 0-based
bt_valid['Topic'] = bt_valid['Topic'] + 1
# Remap integer years back to window-label strings so downstream cells are unaffected
bt_valid['Timestamp'] = bt_valid['Timestamp'].map(YEAR_TO_WINDOW)

# Dominant topic per window (highest frequency) — values are now 1-based
bt_dom_per_window = (
    bt_valid.loc[bt_valid.groupby('Timestamp')['Frequency'].idxmax()]
    .set_index('Timestamp')['Topic']
    .to_dict()
)

print("\nTopics over time (sample):")
print(bt_valid.head(12).to_string(index=False))
print(f"\nDominant topic per window: {bt_dom_per_window}")


In [ ]:
# ── Single static plot: topic prevalence (%) per window ───────────────────────
win_order = {w: i for i, w in enumerate(WINDOW_LABELS)}
min_tid   = int(bt_valid['Topic'].min())
topic_ids_sorted = sorted(bt_valid['Topic'].unique())
colors_bt = plt.cm.tab10(np.linspace(0, 1, len(topic_ids_sorted)))

# Normalise: each topic's share of total docs assigned to any topic in that window
window_totals = bt_valid.groupby('Timestamp')['Frequency'].sum()
bt_pct = bt_valid.copy()
bt_pct['Prevalence'] = bt_pct.apply(
    lambda r: r['Frequency'] / window_totals[r['Timestamp']] * 100, axis=1
)

fig, ax = plt.subplots(figsize=(10, 5))
for i, tid in enumerate(topic_ids_sorted):
    topic_data = (
        bt_pct[bt_pct['Topic'] == tid]
        .copy()
        .assign(_order=lambda d: d['Timestamp'].map(win_order))
        .sort_values('_order')
    )
    top3 = ', '.join([w for w, _ in topic_model.get_topic(int(tid) - min_tid)[:3]])
    ax.plot(topic_data['Timestamp'], topic_data['Prevalence'],
            marker='o', label=f"T{i+1}: {top3}", color=colors_bt[i])

ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0f}%'))
ax.set_title("BERTopic: Topic Prevalence per Time Window", fontsize=12, fontweight='bold')
ax.set_xlabel("Time Window")
ax.set_ylabel("Prevalence (% of window docs)")
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/03_bertopic_over_time.png", dpi=150, bbox_inches='tight')
plt.show()

### 6c. Topic prevalence table over time

In [ ]:
pivot_bt = bt_valid.pivot_table(index='Topic', columns='Timestamp',
                                values='Frequency', aggfunc='sum').fillna(0)

ordered_cols = [c for c in WINDOW_LABELS if c in pivot_bt.columns]
pivot_bt = pivot_bt[ordered_cols]
pivot_bt_norm = pivot_bt.div(pivot_bt.sum(axis=0), axis=1)

# Build display table — all windows including 2015-2016 for completeness
tot_out = bt_valid[['Topic', 'Words', 'Frequency', 'Timestamp']].copy()
tot_out = tot_out.sort_values(['Timestamp', 'Topic']).reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12, len(tot_out) * 0.38 + 1))
ax.axis('off')

col_labels = ['Topic', 'Words', 'Frequency', 'Window']
cell_text = [
    [str(int(r.Topic)), r.Words, str(int(r.Frequency)), r.Timestamp]
    for r in tot_out.itertuples()
]

tbl = ax.table(
    cellText=cell_text,
    colLabels=col_labels,
    cellLoc='left',
    loc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.auto_set_column_width([0, 1, 2, 3])

# Header styling
for j in range(len(col_labels)):
    tbl[0, j].set_facecolor('#2c3e50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')

# Alternating row shading per window
window_order = {w: i for i, w in enumerate(WINDOW_LABELS)}
for i, r in enumerate(tot_out.itertuples(), start=1):
    shade = '#eaf2fb' if window_order.get(r.Timestamp, 0) % 2 == 0 else '#fdfefe'
    for j in range(len(col_labels)):
        tbl[i, j].set_facecolor(shade)

plt.suptitle('BERTopic: Topics Over Time', fontsize=12, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/06b_bertopic_topics_over_time.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {PLOT_DIR}/06b_bertopic_topics_over_time.png")

## 7. Comparative Visualizations

In [ ]:
def generate_wordcloud(words, weights=None, title='', ax=None, colormap='Blues'):
    if weights is not None:
        freq_dict = dict(zip(words, weights))
    else:
        freq_dict = {w: 1 for w in words}
    wc = WordCloud(width=400, height=200, background_color='white',
                   colormap=colormap, max_words=50).generate_from_frequencies(freq_dict)
    if ax:
        ax.imshow(wc, interpolation='bilinear')
        ax.axis('off')
        ax.set_title(title, fontsize=8, fontweight='bold')


def top_terms_from_docs(docs, max_terms=20):
    if len(docs) == 0:
        return [], []
    try:
        vec = CountVectorizer(stop_words='english', min_df=1, ngram_range=(1, 2))
        mat = vec.fit_transform(docs)
    except ValueError:
        return [], []
    scores = np.asarray(mat.sum(axis=0)).ravel()
    top_idx = scores.argsort()[::-1][:max_terms]
    vocab_local = vec.get_feature_names_out()
    return [vocab_local[i] for i in top_idx], scores[top_idx]


def lda_window_weighted_terms(win, dom_topic, max_terms=20):
    win_idx = temporal_reset.index[temporal_reset['time_window'] == win].to_numpy()
    if len(win_idx) == 0:
        return [], []
    topic_weights = doc_topic_global[win_idx, dom_topic]
    weighted_counts = np.asarray(dtm_global[win_idx].T.dot(topic_weights)).ravel()
    top_idx = weighted_counts.argsort()[::-1][:max_terms]
    return [vocab[i] for i in top_idx], weighted_counts[top_idx]


windows_lda_bt = [w for w in WINDOW_LABELS if w in lda_results]
n_windows = len(windows_lda_bt)

fig, axes = plt.subplots(2, n_windows, figsize=(4 * n_windows, 5))
if n_windows == 1:
    axes = axes.reshape(2, 1)

for j, win in enumerate(windows_lda_bt):
    # LDA dominant topic, visualised with window-level weighted word counts.
    mean_weights  = lda_results[win]['doc_topic_matrix'].mean(axis=0)
    dom_lda       = np.argmax(mean_weights)
    lda_words, lda_top_w = lda_window_weighted_terms(win, dom_lda, max_terms=20)
    generate_wordcloud(lda_words, lda_top_w,
                       title=f"LDA T{dom_lda+1} [{win}]",
                       ax=axes[0, j], colormap='Blues')

    # BERTopic dominant topic, visualised from docs in this window assigned to that topic.
    dom_bt    = bt_dom_per_window.get(win, 1)
    topic_ids = np.asarray(bt_topics)
    win_mask = (temporal_reset['time_window'] == win).to_numpy()
    bt_docs = temporal_reset.loc[win_mask & (topic_ids == dom_bt - 1), 'clean_text'].tolist()
    bt_words, bt_scores = top_terms_from_docs(bt_docs, max_terms=20)
    if len(bt_words) == 0:
        bt_row = bt_valid[(bt_valid['Timestamp'] == win) & (bt_valid['Topic'] == dom_bt)]
        bt_words = [w.strip() for w in bt_row.iloc[0]['Words'].split(',')] if len(bt_row) else []
        bt_scores = None
    generate_wordcloud(bt_words, bt_scores,
                       title=f"BERTopic T{dom_bt} [{win}]",
                       ax=axes[1, j], colormap='Oranges')

axes[0, 0].set_ylabel('LDA', fontsize=9)
axes[1, 0].set_ylabel('BERTopic', fontsize=9)
plt.suptitle('Dominant Topic WordCloud: LDA vs BERTopic per Window',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/05_dominant_topic_wordclouds.png", dpi=150, bbox_inches='tight')
plt.show()

### 7b. Side-by-side top-terms comparison: LDA vs BERTopic (fake news only)

In [ ]:
# Both models operate on temporal_df filtered to majority_target == 'False'
print("Side-by-side comparison: Dominant topic top terms per window (fake news only)")
print("=" * 80)
print(f"{'Window':<15} {'LDA Dominant Topic':<35} {'BERTopic Dominant Topic'}")
print("-" * 80)

for win in windows_lda_bt:
    mean_w    = lda_results[win]['doc_topic_matrix'].mean(axis=0)
    dom_lda   = np.argmax(mean_w)
    lda_words, _ = lda_window_weighted_terms(win, dom_lda, max_terms=6)
    lda_terms = ', '.join(lda_words)

    # Use window-level docs assigned to the dominant BERTopic topic.
    dom_bt   = bt_dom_per_window.get(win, 1)
    topic_ids = np.asarray(bt_topics)
    win_mask = (temporal_reset['time_window'] == win).to_numpy()
    bt_docs = temporal_reset.loc[win_mask & (topic_ids == dom_bt - 1), 'clean_text'].tolist()
    bt_words, _ = top_terms_from_docs(bt_docs, max_terms=6)
    if len(bt_words) == 0:
        bt_row = bt_valid[(bt_valid['Timestamp'] == win) & (bt_valid['Topic'] == dom_bt)]
        bt_words = [w.strip() for w in bt_row.iloc[0]['Words'].split(',')[:6]] if len(bt_row) else []
    bt_terms = ', '.join(bt_words)

    print(f"{win:<15} {lda_terms:<35} BERTopic T{dom_bt}: {bt_terms}")